In [5]:
# !pip install --upgrade pip

# # Torch estable
# !pip install torch==2.1.1 --index-url https://download.pytorch.org/whl/cu121

# # Ecosistema HF estable
# !pip install --upgrade transformers
# !pip install --upgrade tokenizers
# !pip install --upgrade datasets
# !pip install --upgrade huggingface_hub
# !pip install --upgrade accelerate
# !pip install --upgrade bitsandbytes
# !pip install --upgrade peft

# # Dependencias extra
# !pip install --upgrade fsspec
# !pip install --upgrade pyarrow
# !pip install --upgrade lxml
# !pip install --upgrade sacrebleu

# !pip install -i https://test.pypi.org/simple/ lowresource-llm-evaluation==0.2.6

In [1]:
import os
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from lowresource_llm_evaluation import benchmark
from lowresource_llm_evaluation.interferenciaLinguistica import loadLexicon
import pandas as pd
import numpy as np
import torch
from huggingface_hub import login
import time
import json
import gc
from dotenv import load_dotenv

base = "./"
load_dotenv("secrets.env")
login()#token=os.getenv("HF_TOKEN"))

/usr/local/lib/python3.11/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /usr/local/lib/python3.11/dist-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
def clean_graphics_card():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    torch.cuda.empty_cache()

def load_gallego():
    with open(base + "EvalDatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
        esp = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
        gl = fGl.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
        espTest = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
        glTest = fGl.readlines()
    return pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])

def evaluate_benchmark(model_name, idioma, token, N= 20, debug=False):
    
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=  torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # 3. Load the pre-trained language model with quantization
    model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                trust_remote_code=True,
                tie_word_embeddings=False, # Added to silence the warning about tied weights
                token = token
            )
    
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id


    print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
    print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
    print(f"Model device: {model.device}")
    codigos = {"aranes": "aran" , 
               "asturiano": "ast", 
               "gallego": "gl"}
    df_textos = {"aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet").sample(N) , 
            "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet").sample(N), 
            "gallego": load_gallego() }
    results = benchmark(model, tokenizer, 
            df_textos = df_textos[idioma],
            lang_eval= codigos[idioma],
            df_huecos=  pd.read_csv(base + f"EvalDatasets/Huecos/{idioma}.csv").sample(N),
            df_anotado = pd.read_csv(base + f"EvalDatasets/Anotado/{idioma}.csv").sample(N),
            lexicon_target = loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"),
            lexicons_comparison = {"es": loadLexicon(base + f"lexicons/es.txt"), "fr": loadLexicon(base + f"lexicons/fr.txt")},
            roundtrip_langs= ["es"],
            debug=debug)
    # Lberamos GPU
    try:
        model.to("cpu")
        del model
        del tokenizer
        clean_graphics_card()
    except Exception as e:
        print("Borrar el modelo ha fallado")
        print(e)
    return results

# Aranés

## Mistral 7B 

In [ ]:
idioma = "aranes"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Tokenizer loaded: TokenizersBackend
Model loaded with 4-bit quantization: MistralForCausalLM
Model device: cuda:0


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Empezando CALIDAD DE LENGUA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

TRADUCCIÓN ROUND TRIP acabado  en 24.77
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 7.57
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 15.11

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.4798356676325758                                    |
| entropy         | 6.8000965826162965                                    |
| ngram_overlap   | 0.0                                                   |
| freq_target     | 0.4707233156654372                                    |
| freq_comparison | {'es': 0.5136248534095068, 'fr': 0.31886916721694664} |
| calidad         | 0.18257673590920848                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Salamandra

In [ ]:
idioma = "aranes"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 9.34
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 1.2
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 4.01

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.7662140182335564                                   |
| entropy         | 7.162224711277058                                    |
| ngram_overlap   | 0.0                                                  |
| freq_target     | 0.43527779958599655                                  |
| freq_comparison | {'es': 0.44054544047666677, 'fr': 0.287624824354361} |
| calidad         | 0.2506109457239275                                   |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor            

## Llama

In [3]:
idioma = "aranes"
modelo = "meta-llama/Llama-3.1-8B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-69c02ae7-4ca8587f643750871374f212;06b1a897-1023-4679-b90d-3807cb538281)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.1-8B-Instruct is awaiting a review from the repo authors.

## Qwen

In [3]:
idioma = "aranes"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Tokenizer loaded: Qwen2Tokenizer
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CALIDAD DE LENGUA acabada en 5.18 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 9.99
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 25.66
Empezando VOCABULARIO
VOCABULARIO acabado  en 8.19
Empezando ORTOGRAFÍA
ORTOGRAFÍA acabado en 16.48

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.3584604994692714                                   |
| entropy         | 6.342755630653786                                    |
| ngram_overlap   | 0.0012518170791552086                                |
| freq_target     | 0.5022136910294805                                   |
| freq_comparison | {'fr': 0.3651248640283728, 'es': 0.5559363438749403} |
| calidad         | 0.11641633453384599

## Llama

In [ ]:
idioma = "aranes"
modelo = "meta-llama/Llama-3.1-8B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-69c02ae7-4ca8587f643750871374f212;06b1a897-1023-4679-b90d-3807cb538281)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.1-8B-Instruct is awaiting a review from the repo authors.

## Qwen

In [ ]:
idioma = "aranes"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Tokenizer loaded: Qwen2Tokenizer
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CALIDAD DE LENGUA acabada en 5.18 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 9.99
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 25.66
Empezando VOCABULARIO
VOCABULARIO acabado  en 8.19
Empezando ORTOGRAFÍA
ORTOGRAFÍA acabado en 16.48

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.3584604994692714                                   |
| entropy         | 6.342755630653786                                    |
| ngram_overlap   | 0.0012518170791552086                                |
| freq_target     | 0.5022136910294805                                   |
| freq_comparison | {'fr': 0.3651248640283728, 'es': 0.5559363438749403} |
| calidad         | 0.11641633453384599

# Asturiano

## Mistral 7B 

In [3]:
idioma = "asturiano"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Tokenizer loaded: TokenizersBackend
Model loaded with 4-bit quantization: MistralForCausalLM
Model device: cuda:0


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Empezando CALIDAD DE LENGUA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

TRADUCCIÓN ROUND TRIP acabado  en 26.57
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 7.57
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 13.83

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.43234389583283306                                   |
| entropy         | 6.789199486639421                                     |
| ngram_overlap   | 0.0004                                                |
| freq_target     | 0.44500516283235836                                   |
| freq_comparison | {'es': 0.6899327339270915, 'fr': 0.28372522746819945} |
| calidad         | 0.1309432543432702                                    |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+-------------------+
| Clave | Valor  

## Salamandra

In [5]:
idioma = "asturiano"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 28.12
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 5.09
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 15.91

                        Evaluación de Calidad de Lengua                         

+-----------------+-----------------------------------------------------+
| Clave           | Valor                                               |
+-----------------+-----------------------------------------------------+
| ttr             | 0.8134602796160053                                  |
| entropy         | 8.645066570260353                                   |
| ngram_overlap   | 0.0007407407407407408                               |
| freq_target     | 0.49889805982662183                                 |
| freq_comparison | {'es': 0.8039235643458522, 'fr': 0.199963405000876} |
| calidad         | 0.13362663018372412                                 |
+-----------------+-----------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor              |
+----

## Llama

In [6]:
idioma = "asturiano"
modelo = "meta-llama/Llama-3.1-8B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct.
403 Client Error. (Request ID: Root=1-69c05869-535666241354160b7666d18a;b22341cf-28d1-4048-9a1e-53baf77a2e7b)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json.
Your request to access model meta-llama/Llama-3.1-8B-Instruct is awaiting a review from the repo authors.

## Qwen

In [4]:
idioma = "asturiano"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Tokenizer loaded: Qwen2Tokenizer
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CALIDAD DE LENGUA acabada en 3.7 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 9.47
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 18.05
Empezando VOCABULARIO
VOCABULARIO acabado  en 6.04
Empezando ORTOGRAFÍA
ORTOGRAFÍA acabado en 10.57

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.41220552494265705                                  |
| entropy         | 6.323841238553982                                    |
| ngram_overlap   | 0.0                                                  |
| freq_target     | 0.5363666522774605                                   |
| freq_comparison | {'es': 0.838617952742621, 'fr': 0.33988943550641093} |
| calidad         | 0.13279799873712714 